# Champaign, Urbana, Savoy Housing Market Analysis

This notebook walks through a hands-on market analysis of single-family homes sold in Champaign, Urbana, and Savoy. We use it as a worked example for the consulting scenario described in the [case overview](./case-overview.md): we are advising *Kingfisher Residential Partners* on where and how to deploy a \$5–10M residential investment.

Our analysis follows five questions, in order:

1. **Market overview** — how do the three cities compare on price and inventory?
2. **Pricing drivers** — what property features explain the variation in price?
3. **Market liquidity** — how quickly are properties selling?
4. **Temporal trends** — is the market accelerating or cooling?
5. **Geographic insights** — where in the metro is the best value?

Each section ends with a short take-away an investor could act on.


## ⚙️ Setup

We lean on a small, focused stack: `pandas` for tabular work, `numpy` for numerics, and `plotly.express` for interactive charts. Plotly is well-suited to a real-estate notebook because hovering over a point lets us see the underlying property details — useful for sanity-checking individual listings.


In [2]:
import pandas as pd
import numpy as np
import plotly
import plotly.express as px

print(f"Pandas version: {pd.__version__}")
print(f"Numpy version: {np.__version__}")
print(f"Plotly version: {plotly.__version__}")

Pandas version: 3.0.0
Numpy version: 2.4.2
Plotly version: 6.5.2


In [4]:
# Set pandas display options to show more columns
pd.set_option("display.max_columns", 50)

## 📥 Load the Dataset

The data is published as a Parquet file on GitHub. Parquet is the format of choice for real-estate panel data because it is columnar (so reading just `price` and `living_area` is fast), it preserves the original dtypes (datetimes stay datetimes), and it is dramatically smaller than CSV.


In [5]:
df = pd.read_parquet(
    "https://github.com/bdi593/datasets/raw/refs/heads/main/zillow-properties/zillow_properties_champaign_urbana_savoy.parquet"
)
df.head(3)

,property_id,date_posted,days_on_zillow,price,description,beds,baths,year_built,latitude,longitude,lot_size,living_area,address_street,address_city,address_state,address_zipcode,reso_has_garage,reso_has_attached_garage,reso_garage_parking_capacity,reso_parking_capacity,reso_attic,reso_stories,reso_basement_yn,reso_has_cooling,reso_has_heating,reso_is_new_construction,reso_elementary_school,reso_middle_or_junior_school,reso_high_school,reso_tax_annual_amount
0,3232534,2022-05-24,1053,165000,This 3 bedroom ranch in an established neighbo...,3,1.0,1958,40.093950,-88.26074,10660.0,1436.0,1804 McDonald Dr,Champaign,IL,61821,True,1.0,1.0,1,NaN,1.0,1.0,1.0,1.0,0.0,Unit 4 Of Choice,Champaign/Middle Call Unit 4 351,Central High School,3270.0
1,3197892,2022-05-27,1028,900000,A TRULY SPECTACULAR HOME IN LINCOLNSHIRE FIELD...,4,6.0,1993,40.095110,-88.30598,32234.0,5430.0,1809 Cobblefield Ct,Champaign,IL,61822,True,1.0,3.0,3,NaN,2.0,1.0,1.0,1.0,0.0,Unit 4 Of Choice,Champaign/Middle Call Unit 4 351,Centennial High School,18782.0
2,3238411,2022-06-15,755,245000,"Wow! 711 W Green is a 5 bedroom, 2 full bath ...",5,2.0,1903,40.110363,-88.21728,6098.0,2046.0,711 W Green St,Urbana,IL,61801,False,0.0,NaN,2,NaN,2.0,1.0,1.0,1.0,0.0,Leal Elementary School,Urbana Middle School,Urbana High School,9413.0


Take a quick look at the schema. Note the mix of property attributes (`beds`, `baths`, `living_area`, `year_built`), location columns (`latitude`, `longitude`, `address_city`, `address_zipcode`), market-activity fields (`date_posted`, `days_on_zillow`), and school information (`reso_high_school`). This breadth is what makes the dataset useful for the five business questions.


In [6]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 3737 entries, 0 to 3736
Data columns (total 30 columns):
 #   Column                        Non-Null Count  Dtype         
---  ------                        --------------  -----         
 0   property_id                   3737 non-null   int64         
 1   date_posted                   3299 non-null   datetime64[us]
 2   days_on_zillow                3737 non-null   int64         
 3   price                         3737 non-null   int64         
 4   description                   3683 non-null   str           
 5   beds                          3737 non-null   int64         
 6   baths                         3736 non-null   float64       
 7   year_built                    3737 non-null   int64         
 8   latitude                      3716 non-null   float64       
 9   longitude                     3716 non-null   float64       
 10  lot_size                      2998 non-null   float64       
 11  living_area                   3725 non-nu

## 🏙️ 1. Market Overview

### Price Distribution by City

A box plot is the standard first look at price across geographies. The box shows the middle 50% of prices (P25 → P75), the line inside is the median, and the dots are individual listings the chart treats as outliers.


In [7]:
fig = px.box(
    df,
    y="price",
    x="address_city",
    title="Price Distribution by City",
    template="simple_white",
)
fig.show()

:::{tip} Why median, not mean?
Real-estate prices are right-skewed: a single \$2M custom home will pull the *mean* sharply higher without being representative of the typical buyer's experience. The *median* is what every CMA, every Zillow market report, and every appraiser uses for this reason.
:::

### Price per Square Foot

Raw price tells us about a property; **price per square foot** lets us compare properties of different sizes on the same yard-stick. It is the single most-used unit in real estate analytics.


In [9]:
df["price_per_sqft"] = df["price"] / df["living_area"]
df[["price", "living_area", "price_per_sqft"]].head(3)

,price,living_area,price_per_sqft
0,165000,1436.0,114.902507
1,900000,5430.0,165.745856
2,245000,2046.0,119.745846


Notice that the distributions for the three cities overlap heavily, even though their *median total prices* differ — this is because Savoy and parts of Urbana have larger lots and bigger homes, not necessarily more expensive *space*.


In [11]:
fig = px.box(
    df,
    x="price_per_sqft",
    y="address_city",
    title="Price per Sqft by City",
    template="simple_white",
)
fig.show()

### Median Price by City

A bar chart of median prices makes the city comparison crisp. For an investor, this is the headline of the market overview slide.


In [18]:
df_city_summary = (
    df.groupby("address_city")
    .agg(
        count=("property_id", "count"),
        median_price=("price", "median"),
        avg_price_per_sqft=("price_per_sqft", "mean"),
    )
    .reset_index()
)

fig1 = px.bar(
    df_city_summary,
    x="address_city",
    y="median_price",
    text_auto=".2s",
    title="Median Listing Price by City",
    labels={"median_price": "Median Price ($)", "address_city": "City"},
    template="simple_white",
    color="address_city",
)
fig1.show()

## 💰 2. Pricing Drivers

With the high-level picture in mind, we can ask: *which property features actually drive price?* We start with a correlation matrix on the obvious numeric candidates.


In [19]:
num_cols = [
    "price",
    "beds",
    "baths",
    "living_area",
    "year_built",
    "reso_tax_annual_amount",
]
corr = df[num_cols].corr()

fig = px.imshow(
    corr,
    text_auto=".2f",
    title="Price Drivers: Correlation Matrix",
    color_continuous_scale="RdBu_r",
    template="simple_white",
)
fig.show()

:::{caution} Correlation ≠ causation
The matrix above tells us which features *move together* with price. It does not tell us that adding a bathroom causes a price jump — bigger houses tend to have more bathrooms *and* sell for more, so the correlation captures both effects.
:::

### Living Area vs Price

Living area is almost always the single strongest predictor of price. The OLS trend line on each city makes the slope — *roughly the implied price per square foot* — directly visible.


In [23]:
# Price vs Living Area
fig = px.scatter(
    df,
    x="living_area",
    y="price",
    color="address_city",
    trendline="ols",
    title="Relationship: Living Area vs Price",
    labels={"living_area": "Living Area (sqft)", "price": "Price ($)"},
    hover_data=["address_street"],
    template="simple_white",
)
fig.update_traces(marker=dict(size=3))
fig.update_yaxes(range=[0, 1500000])
fig.show()

If we strip out the city colouring and the trend line, we can see the raw cloud of points. The fan-out at higher square-footage is classic: large luxury homes have very wide pricing because location and finish matter much more at the high end.


In [15]:
fig = px.scatter(
    df,
    x="living_area",
    y="price",
    opacity=0.4,
    title="Price vs Living Area",
    template="simple_white",
)
fig.update_yaxes(range=[0, 1000000])
fig.show()

### Bedrooms vs Price

Bedroom count is correlated with size, but it also has a value of its own — a 4-bedroom house at the same square-footage as a 3-bedroom typically commands a premium because it appeals to a larger family-buyer pool.


In [16]:
fig = px.box(
    df,
    x="beds",
    y="price",
    title="Price by Number of Bedrooms",
    template="simple_white",
)
fig.show()

### The Garage Premium

In the Midwest, an attached garage is close to a hard requirement for most buyers. The chart below quantifies the price gap between listings with and without a garage. For an investor, this is one of the higher-ROI capex items to evaluate when considering a fixer-upper.


In [45]:
fig = px.box(
    df, x="price", y="reso_has_garage", title="Garage vs Price", template="simple_white"
)
fig.update_xaxes(range=[0, 2000000])
fig.show()

## ⏱️ 3. Market Liquidity

*Days on Zillow* is the headline liquidity metric. A short days-on-market distribution means properties are turning fast — a *seller's market*. A long, fat right tail signals stale listings and bargaining power for buyers.


In [43]:
fig = px.histogram(
    df,
    x="days_on_zillow",
    color="address_city",
    facet_col="address_city",
    title="Market Liquidity: Days on Zillow by City",
    labels={"days_on_zillow": "Days on Market", "address_city": "City"},
    template="simple_white",
)

fig.update_layout(showlegend=False)
fig.show()

:::{seealso} Days-on-market and offer strategy
A market where the median days-on-market is below 14 typically means buyers should expect bidding wars and offer at or above asking. Above 60 days, sellers are usually open to concessions and price reductions.
:::

## 📈 4. Temporal Trends

Next, we look at how prices and inventory have changed month over month. We aggregate `date_posted` into year-month buckets and compute the median price for each bucket.


In [35]:
df_temporal = (
    df.groupby(df["date_posted"].dt.strftime("%Y-%m"))
    .agg(
        median_price=("price", "median"),
        count=("property_id", "count"),
    )
    .reset_index()
    .rename(columns={"date_posted": "month_year"})
)

df_temporal.head(3)

,month_year,median_price,count
0,2022-05,532500.0,2
1,2022-06,245000.0,3
2,2022-07,415000.0,1


Plotting the resulting time series shows whether the local market is heating up, cooling down, or sideways. For an acquirer like Kingfisher, the slope of this line — combined with the days-on-market distribution above — informs how aggressive their bid strategy should be.


In [39]:
fig = px.line(
    df_temporal,
    x="month_year",
    y="median_price",
    markers=True,
    title="Median Price Trends Over Time",
    labels={"median_price": "Median Price ($)", "month_year": "Date"},
    template="simple_white",
)
fig.show()

## 🗺️ 5. Geographic Insights

Finally, we map every listing and colour it by **price per square foot**. This is the most actionable single visual in the notebook: it shows *where in the metro* you get the most house for your money.

Look for cool-coloured (low \$/sqft) clusters near amenities — those are the highest-upside acquisition zones for a value investor.


In [55]:
fig = px.scatter_map(
    df,
    lat="latitude",
    lon="longitude",
    color="price_per_sqft",
    hover_name="address_street",
    title="Geographic Value Map (Color=$/sqft)",
    color_continuous_scale=px.colors.sequential.ice,
    zoom=11,
    template="simple_white",
)
fig.show()

### School Premium

School quality is one of the strongest sub-market drivers in U.S. residential real estate. We approximate it here by computing the median sale price under each high-school catchment.


In [ ]:
df_school_prices = (
    df.groupby("reso_high_school")["price"]
    .median()
    .sort_values(ascending=False)
    .head(10)
    .reset_index()
)

:::{important} Caveat: confounding
High-priced school catchments are *also* often high-amenity neighborhoods, so the chart above bundles "good schools" with "good shopping", "low crime", "older trees", etc. Untangling these effects rigorously requires controlling for property characteristics — which is exactly what the regression and ML notebooks in other chapters teach.
:::


In [59]:
fig = px.bar(
    df_school_prices,
    x="price",
    y="reso_high_school",
    title="Top Schools by Median Price",
    template="simple_white",
)
fig.update_yaxes(categoryorder="total ascending")
fig.show()

## 📌 Recap & Recommendation for Kingfisher

Pulling the threads together, our analysis suggests:

- **Pricing posture.** Prices in Champaign-Urbana-Savoy have been broadly stable, with a modest upward drift. There is no urgency to over-bid.
- **Highest-value cities.** Savoy commands the highest median prices but also the highest \$/sqft. Champaign offers the largest pool of listings and the broadest \$/sqft range — meaning more room for value plays.
- **Feature focus.** Living area, bathrooms, and garage presence dominate the price drivers. A targeted renovation strategy that adds a bathroom or attached garage is likely to deliver attractive ROI.
- **Market liquidity.** Most properties sell within a few months, but a non-trivial tail of listings sits for 100+ days. Those stale listings are negotiation opportunities.
- **Geography & schools.** Premium high-school catchments command a clear premium. For Kingfisher's rental thesis, the *next-best* catchments often offer a better cap-rate at the cost of slightly slower appreciation.

These findings are descriptive only. The next step in a real consulting engagement would be to fit a regression or gradient-boosted model to estimate fair value for each property and flag *under-priced* listings to the acquisition team — the same modelling pattern we work through in the *Australian Rental Market* chapter.
